In [ ]:
# =========================================================
# 1. INSTALL + IMPORTS + DRIVE MOUNT
# =========================================================
#!pip install -q -U transformers datasets peft bitsandbytes accelerate trl evaluate scikit-learn
!pip uninstall -y torch torchvision torchaudio transformers peft trl accelerate bitsandbytes -q
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers==4.48.1 accelerate==1.3.0 peft==0.14.0 trl datasets bitsandbytes
!pip install -q scikit-learn pandas numpy sentencepiece


from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training
)
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_MODEL_ID = "microsoft/Phi-4-mini-instruct"
DATASET_ID = "openlifescienceai/medmcqa"

DRIVE_ROOT = "/content/drive/MyDrive/phi4_medmcqa"
QLORA_SAVE_DIR = os.path.join(DRIVE_ROOT, "qlora_adapter_1")
LORA_SAVE_DIR = os.path.join(DRIVE_ROOT, "lora_adapter_1")
METRICS_DIR = os.path.join(DRIVE_ROOT, "metrics_1")

os.makedirs(QLORA_SAVE_DIR, exist_ok=True)
os.makedirs(LORA_SAVE_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 110.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 140.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 47.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 116.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# =========================================================
# 2. LOAD RAW DATASET
# =========================================================
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load the dataset
dataset_name = "openlifescienceai/medmcqa"
print(f"Loading dataset: {dataset_name}...")
dataset = load_dataset(dataset_name)

# Print the structure of the dataset
print("\nDataset Structure:")
print(dataset)

# Corrected lines to use the 'dataset' object instead of 'dataset_name'
print(dataset) # Changed from dataset_name to dataset
print(dataset["train"].column_names) # Changed from dataset_name to dataset
print(dataset["train"][0]) # Changed from dataset_name to dataset

Loading dataset: openlifescienceai/medmcqa...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]


Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 4183
    })
})
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question',

In [ ]:
# =========================================================
# 3. BASIC EDA
# =========================================================
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nMissing values in train:")
print(train_df[["question", "opa", "opb", "opc", "opd", "cop"]].isnull().sum())

print("\nLabel distribution in train:")
print(train_df["cop"].value_counts().sort_index())

label_map = {0: "A", 1: "B", 2: "C", 3: "D"}
train_df["answer_letter"] = train_df["cop"].map(label_map)
val_df["answer_letter"] = val_df["cop"].map(label_map)

print("\nTrain label distribution:")
print(train_df["answer_letter"].value_counts())

Train shape: (182822, 11)
Validation shape: (4183, 11)

Missing values in train:
question    0
opa         0
opb         0
opc         0
opd         0
cop         0
dtype: int64

Label distribution in train:
cop
0    53591
1    47826
2    42442
3    38963
Name: count, dtype: int64

Train label distribution:
answer_letter
A    53591
B    47826
C    42442
D    38963
Name: count, dtype: int64


In [ ]:
# =========================================================
# 4. PROMPT LENGTH CHECK
# =========================================================
def build_prompt_only(example):
    return f"""You are a medical expert answering a multiple choice medical question.
Read the question carefully and select the single best option.
Reply with only one letter: A, B, C, or D.

Question: {example['question']}

Options:
A. {example['opa']}
B. {example['opb']}
C. {example['opc']}
D. {example['opd']}

Answer:
"""

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample_prompts = [build_prompt_only(dataset["train"][i]) for i in range(500)]
lengths = [len(tokenizer(p)["input_ids"]) for p in sample_prompts]

print("Min tokens:", min(lengths))
print("Max tokens:", max(lengths))
print("Mean tokens:", sum(lengths)/len(lengths))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Min tokens: 67
Max tokens: 246
Mean tokens: 91.44


In [ ]:
# =========================================================
# 5. FORMAT DATA FOR TRAINING
# =========================================================
label_map = {0: "A", 1: "B", 2: "C", 3: "D"}

def format_for_sft(example):
    # This check is no longer strictly needed here if we filter first, but good for robustness
    # if example["cop"] not in label_map:
    #     return None # Or handle as appropriate

    answer_letter = label_map[example["cop"]]
    text = f"""You are a medical expert answering a multiple choice medical question.
Read the question carefully and select the single best option.
Reply with only one letter: A, B, C, or D.

Question: {example['question']}

Options:
A. {example['opa']}
B. {example['opb']}
C. {example['opc']}
D. {example['opd']}

Answer:
{answer_letter}"""
    return {"text": text}

# Filter out examples where 'cop' is -1 or any other invalid value before mapping
filtered_dataset = dataset.filter(lambda example: example["cop"] in label_map)
formatted_dataset = filtered_dataset.map(format_for_sft)
print(formatted_dataset["train"][0]["text"])

Filter:   0%|          | 0/182822 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4183 [00:00<?, ? examples/s]

Map:   0%|          | 0/182822 [00:00<?, ? examples/s]

Map:   0%|          | 0/4183 [00:00<?, ? examples/s]

You are a medical expert answering a multiple choice medical question.
Read the question carefully and select the single best option.
Reply with only one letter: A, B, C, or D.

Question: Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma

Options:
A. Hyperplasia
B. Hyperophy
C. Atrophy
D. Dyplasia

Answer:
C


In [ ]:
# =========================================================
# 6. LOAD MODEL FOR QLORA
# =========================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qlora_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

qlora_model.config.use_cache = False
qlora_model = prepare_model_for_kbit_training(qlora_model)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
# =========================================================
# 7. QLORA TRAINING
# =========================================================

#train_sample = formatted_dataset["train"].shuffle(seed=42).select(range(30000)) # Iteration 1
train_sample = formatted_dataset["train"].shuffle(seed=42).select(range(60000)) # Iteration 2
eval_sample = formatted_dataset["validation"].shuffle(seed=42).select(range(1000))

print("Train sample size:", len(train_sample))
print("Eval sample size :", len(eval_sample))

qlora_args = SFTConfig(
    output_dir=QLORA_SAVE_DIR,
    dataset_text_field="text",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    #learning_rate=2e-4, # Iteration 1
    learning_rate=1e-4, # Iteration 2
    num_train_epochs=1,

    # this will show logs during training
    logging_strategy="steps",
    logging_steps=50,

    # this will run validation during training
    eval_strategy="steps",
    eval_steps=200,

    # this will save checkpoints during training
    save_strategy="steps",
    save_steps=200,

    fp16=True,
    bf16=False,
    report_to="none",
    max_length=512
)

qlora_trainer = SFTTrainer(
    model=qlora_model,
    train_dataset=train_sample,
    eval_dataset=eval_sample,
    processing_class=tokenizer,
    args=qlora_args,
    peft_config=qlora_config
)

print("Starting QLoRA training...")
qlora_train_result = qlora_trainer.train()
print("Training finished.")

Train sample size: 60000
Eval sample size : 1000


Converting train dataset to ChatML:   0%|          | 0/60000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/60000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/60000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/60000 [00:00<?, ? examples/s]

...

Starting QLoRA training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,0.893200,1.022344
400,0.859000,1.015819
600,0.849300,1.015295
800,0.855900,1.013089
1000,0.885700,1.008559
1200,0.857600,1.008867
1400,0.865200,1.008088
1600,0.864500,1.006520
1800,0.841100,1.007036
2000,0.873000,1.004754


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

Training finished.


In [ ]:
# =========================================================
# 8. SAVE QLORA MODEL + TOKENIZER TO DRIVE
# =========================================================
qlora_trainer.model.save_pretrained(QLORA_SAVE_DIR)
tokenizer.save_pretrained(QLORA_SAVE_DIR)

with open(os.path.join(METRICS_DIR, "qlora_train_result.json"), "w") as f:
    json.dump(qlora_train_result.metrics, f, indent=2)

print("QLoRA adapter saved to:", QLORA_SAVE_DIR)

QLoRA adapter saved to: /content/drive/MyDrive/phi4_medmcqa/qlora_adapter_1


In [ ]:
# =========================================================
# 9. COMMON EVALUATION HELPERS
# =========================================================
def build_eval_prompt(example):
    return f"""You are a medical expert answering a multiple choice medical question.
Read the question carefully and select the single best option.
Reply with only one letter: A, B, C, or D.

Question: {example['question']}

Options:
A. {example['opa']}
B. {example['opb']}
C. {example['opc']}
D. {example['opd']}

Answer:
"""

def extract_option(text):
    text = text.strip().upper()
    for ch in text:
        if ch in ["A", "B", "C", "D"]:
            return ch
    return "INVALID"

def predict_letter(model, tokenizer, example, max_new_tokens=3):
    prompt = build_eval_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extract_option(generated)

def compute_metrics_from_preds(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [ ]:
# =========================================================
# 10. EVALUATE QLORA ON FULL VALIDATION SET
#     (NO generate() -- use next-token logits)
# =========================================================

import os
import json
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# -----------------------------
# 1. Load raw dataset
# -----------------------------
raw_dataset = load_dataset("openlifescienceai/medmcqa")
label_map = {0: "A", 1: "B", 2: "C", 3: "D"}

# -----------------------------
# 2. Load tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -----------------------------
# 3. Load base model + QLoRA adapter
# -----------------------------
eval_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qlora_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=eval_bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

qlora_eval_model = PeftModel.from_pretrained(qlora_base, QLORA_SAVE_DIR)
qlora_eval_model.eval()

# -----------------------------
# 4. Prompt builder
# -----------------------------
def build_eval_prompt(example):
    return f"""You are a medical expert answering a multiple choice medical question.
Read the question carefully and select the single best option.
Reply with only one letter: A, B, C, or D.

Question: {example['question']}

Options:
A) {example['opa']}
B) {example['opb']}
C) {example['opc']}
D) {example['opd']}

Answer:
"""

# -----------------------------
# 5. Prepare token ids for A/B/C/D
#    We score the NEXT token after "Answer:"
# -----------------------------
candidate_letters = ["A", "B", "C", "D"]
candidate_token_ids = {}

for letter in candidate_letters:
    ids = tokenizer.encode(letter, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Token for '{letter}' is not a single token: {ids}")
    candidate_token_ids[letter] = ids[0]

print("Candidate token ids:", candidate_token_ids)

# -----------------------------
# 6. Prediction via next-token logits
# -----------------------------
def predict_letter_from_logits(model, tokenizer, example, max_length=512):
    prompt = build_eval_prompt(example)

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False
    )

    input_ids = enc["input_ids"].to(qlora_eval_model.device)
    attention_mask = enc["attention_mask"].to(qlora_eval_model.device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    # logits shape: [batch, seq_len, vocab_size]
    next_token_logits = outputs.logits[0, -1, :]   # logits for token right after "Answer:"

    scores = {
        letter: next_token_logits[token_id].item()
        for letter, token_id in candidate_token_ids.items()
    }

    pred = max(scores, key=scores.get)
    return pred, scores

# -----------------------------
# 7. Metric helper
# -----------------------------
def compute_metrics_from_preds(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# -----------------------------
# 8. Run evaluation
# -----------------------------
y_true = []
y_pred = []

validation_data = raw_dataset["validation"]

print(f"Evaluating QLoRA on full validation set: {len(validation_data)} samples")

for i, ex in enumerate(tqdm(validation_data)):
    true_label = label_map[ex["cop"]]
    pred_label, scores = predict_letter_from_logits(qlora_eval_model, tokenizer, ex)

    y_true.append(true_label)
    y_pred.append(pred_label)

    if i < 5:
        print(f"Sample {i} | Pred: {pred_label} | GT: {true_label} | Scores: {scores}")

# -----------------------------
# 9. Final metrics
# -----------------------------
qlora_metrics = compute_metrics_from_preds(y_true, y_pred)

print("\nQLoRA Validation Metrics:")
print(qlora_metrics)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))

# -----------------------------
# 10. Save metrics
# -----------------------------
os.makedirs(METRICS_DIR, exist_ok=True)

with open(os.path.join(METRICS_DIR, "qlora_full_validation_metrics.json"), "w") as f:
    json.dump(qlora_metrics, f, indent=2)

print("\nSaved metrics to:")
print(os.path.join(METRICS_DIR, "qlora_full_validation_metrics.json"))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Candidate token ids: {'A': 32, 'B': 33, 'C': 34, 'D': 35}
Evaluating QLoRA on full validation set: 4183 samples


  0%|          | 1/4183 [00:00<07:09,  9.73it/s]

Sample 0 | Pred: A | GT: A | Scores: {'A': 46.5625, 'B': 43.46875, 'C': 42.1875, 'D': 44.90625}


  0%|          | 2/4183 [00:00<07:23,  9.42it/s]

...

Sample 2 | Pred: C | GT: C | Scores: {'A': 45.15625, 'B': 44.1875, 'C': 45.9375, 'D': 45.03125}


  0%|          | 4/4183 [00:00<07:12,  9.66it/s]

Sample 3 | Pred: C | GT: C | Scores: {'A': 47.0625, 'B': 45.09375, 'C': 47.25, 'D': 42.84375}
Sample 4 | Pred: A | GT: A | Scores: {'A': 46.25, 'B': 44.96875, 'C': 45.0625, 'D': 44.53125}


100%|██████████| 4183/4183 [06:46<00:00, 10.29it/s]


QLoRA Validation Metrics:
{'accuracy': 0.5462586660291656, 'precision_macro': 0.5473114886682792, 'recall_macro': 0.5367321598816287, 'f1_macro': 0.5404360143704909}

Classification Report:
              precision    recall  f1-score   support

           A       0.55      0.62      0.59      1348
           B       0.52      0.52      0.52      1085
           C       0.55      0.53      0.54       925
           D       0.57      0.47      0.52       825

    accuracy                           0.55      4183
   macro avg       0.55      0.54      0.54      4183
weighted avg       0.55      0.55      0.54      4183


Saved metrics to:
/content/drive/MyDrive/phi4_medmcqa/metrics_1/qlora_full_validation_metrics.json


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
from google.colab import runtime
runtime.unassign()